# RAG Pipeline with OpenAI + Pinecone

This notebook demonstrates a **Retrieval-Augmented Generation (RAG)** pipeline using:
- **OpenAI** – `text-embedding-3-small` for embeddings and `gpt-4o-mini` as the chat LLM.
- **Pinecone** – vector database for storing and retrieving document embeddings.
- **LangChain v0.2+** – orchestration via LCEL (LangChain Expression Language).

## Pipeline Overview

```
TXT files  →  TextLoader  →  RecursiveCharacterTextSplitter
                                        ↓
                            OpenAIEmbeddings (text-embedding-3-small)
                                        ↓
                            PineconeVectorStore (upsert)
                                        ↓
                        VectorStoreRetriever (top-k similarity search)
                                        ↓
                    ChatPromptTemplate + ChatOpenAI (gpt-4o-mini)
                                        ↓
                                  Final Answer
```

## Prerequisites

1. Install dependencies: `pip install -r requirements.txt`
2. Copy `.env.example` to `.env` and fill in your API keys.
3. Create a Pinecone index with **dimension=1536** and **metric=cosine** (see Step 3 below).

## Step 1 – Load Environment Variables

We use `python-dotenv` to load secrets from a `.env` file.  
Copy `.env.example` → `.env` and fill in your credentials before running this cell.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the project root (one level above notebooks/)
env_path = Path("../.env")
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅  Loaded environment variables from {env_path.resolve()}")
else:
    load_dotenv()  # fall back to environment variables already set in the shell
    print("ℹ️  No .env file found – using shell environment variables.")

# Verify required variables are present
required_vars = ["OPENAI_API_KEY", "PINECONE_API_KEY", "PINECONE_INDEX_NAME"]
missing = [v for v in required_vars if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing required environment variables: {missing}")

OPENAI_API_KEY    = os.environ["OPENAI_API_KEY"]
PINECONE_API_KEY  = os.environ["PINECONE_API_KEY"]
INDEX_NAME        = os.environ["PINECONE_INDEX_NAME"]
NAMESPACE         = os.getenv("PINECONE_NAMESPACE", "default")

print(f"📌  Pinecone index : {INDEX_NAME}")
print(f"📌  Namespace      : {NAMESPACE}")

## Step 2 – Load TXT Documents

We use LangChain's `DirectoryLoader` + `TextLoader` to load every `.txt` file under `data/`.  
Each file becomes a `Document` with `page_content` (raw text) and `metadata` (including the `source` path).

In [ ]:
import sys
sys.path.insert(0, str(Path("..")))

from src.rag_utils import load_txt_documents

DATA_DIR = "../data"

documents = load_txt_documents(DATA_DIR)

# Preview the first document
print("\n--- First document preview ---")
print(f"Source : {documents[0].metadata['source']}")
print(f"Length : {len(documents[0].page_content)} characters")
print("\nFirst 300 characters:")
print(documents[0].page_content[:300], "...")

## Step 3 – Split Documents into Chunks

`RecursiveCharacterTextSplitter` splits long documents into smaller, overlapping chunks.  
Overlap ensures context is not lost at chunk boundaries.

| Parameter      | Value | Meaning                                          |
|---------------|-------|--------------------------------------------------|
| `chunk_size`  | 1000  | Maximum characters per chunk                    |
| `chunk_overlap`| 200  | Characters shared between consecutive chunks    |

In [ ]:
from src.rag_utils import split_documents

CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 200

chunks = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

print(f"\nTotal chunks : {len(chunks)}")
print("\n--- Sample chunk ---")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

## Step 4 – Create / Verify the Pinecone Index

The embedding model `text-embedding-3-small` produces **1536-dimensional** vectors.  
Your Pinecone index must be created with the matching dimension **before** running the upsert step.

### Option A – Create via Pinecone Console (recommended)
1. Go to [https://app.pinecone.io](https://app.pinecone.io).
2. Click **Create index**.
3. Set **Name** = value of `PINECONE_INDEX_NAME`, **Dimensions** = `1536`, **Metric** = `cosine`.
4. Click **Create index** and wait until the status shows **Ready**.

### Option B – Create programmatically (run the cell below)

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

existing_indexes = [idx.name for idx in pc.list_indexes()]
print(f"Existing indexes: {existing_indexes}")

if INDEX_NAME not in existing_indexes:
    print(f"Creating index '{INDEX_NAME}' ...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,        # must match text-embedding-3-small output dimension
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    # Wait until the index is ready
    import time
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        print("  Waiting for index to be ready...")
        time.sleep(5)
    print(f"✅  Index '{INDEX_NAME}' is ready.")
else:
    print(f"✅  Index '{INDEX_NAME}' already exists.")

## Step 5 – Embed Documents and Upsert into Pinecone

This step:
1. Creates an `OpenAIEmbeddings` instance using `text-embedding-3-small`.
2. Embeds each chunk and upserts the vectors into the Pinecone index.

> **Note:** If you have already upserted documents and want to skip re-ingestion, jump directly to **Step 6**.

In [ ]:
from src.rag_utils import build_vectorstore

EMBEDDING_MODEL = "text-embedding-3-small"

vectorstore = build_vectorstore(
    chunks=chunks,
    index_name=INDEX_NAME,
    namespace=NAMESPACE,
    embedding_model=EMBEDDING_MODEL,
)

print("\n✅  Vectors upserted successfully.")

## Step 6 – (Optional) Connect to an Existing Vectorstore

If you have already upserted your documents in a previous run, use this cell instead of Step 5  
to connect to the existing index without re-embedding.

In [ ]:
# Uncomment the lines below to skip Step 5 and connect to an already-populated index.

# from src.rag_utils import load_vectorstore
# vectorstore = load_vectorstore(
#     index_name=INDEX_NAME,
#     namespace=NAMESPACE,
#     embedding_model=EMBEDDING_MODEL,
# )
# print("✅  Connected to existing vectorstore.")

## Step 7 – Build the RAG Chain

We wire everything together using **LCEL** (LangChain Expression Language):

```
question
   │
   ├─► retriever  ──► format_docs ──► context
   │                                      │
   └──────────────────────────────► ChatPromptTemplate
                                          │
                                    ChatOpenAI (gpt-4o-mini)
                                          │
                                    StrOutputParser
                                          │
                                       answer (str)
```

In [ ]:
from src.rag_utils import build_rag_chain

LLM_MODEL   = "gpt-4o-mini"
TOP_K       = 4          # number of chunks to retrieve per query
TEMPERATURE = 0.0        # deterministic output

rag_chain, retriever = build_rag_chain(
    vectorstore=vectorstore,
    top_k=TOP_K,
    llm_model=LLM_MODEL,
    temperature=TEMPERATURE,
)

print("✅  RAG chain built.")
print(f"    LLM        : {LLM_MODEL}")
print(f"    Retrieval  : top-{TOP_K} chunks")

## Step 8 – Run RAG Queries

Ask questions! The pipeline will retrieve the most relevant document chunks from Pinecone  
and pass them as context to the LLM to generate a grounded answer.

In [ ]:
def ask(question: str) -> str:
    """Run a RAG query and print the answer."""
    print(f"\n❓ Question: {question}")
    answer = rag_chain.invoke(question)
    print(f"\n💬 Answer:\n{answer}")
    return answer

# Example queries
ask("What is Artificial Intelligence and what are its main application areas?")

In [ ]:
ask("What are the main types of machine learning? Give a brief description of each.")

In [ ]:
ask("What is Retrieval-Augmented Generation (RAG) and why is it useful?")

## Step 9 – Inspect Retrieved Sources

We can call the retriever directly to see which document chunks were fetched for a given query,  
allowing us to verify that the right context is being surfaced.

In [ ]:
query = "What are the limitations of Large Language Models?"

retrieved_docs = retriever.invoke(query)

print(f"🔍  Retrieved {len(retrieved_docs)} chunk(s) for: \"{query}\"\n")
for i, doc in enumerate(retrieved_docs, start=1):
    source = doc.metadata.get("source", "unknown")
    print(f"── Chunk {i} ── source: {source}")
    print(doc.page_content[:300], "...\n")

In [ ]:
# Ask the full RAG question with sources displayed afterwards
question = "What are the limitations of Large Language Models?"
answer = ask(question)

print("\n📚  Sources used:")
for doc in retriever.invoke(question):
    print(" -", doc.metadata.get("source", "unknown"))

## Summary

In this notebook we:

| Step | Action |
|------|--------|
| 1 | Loaded API keys from `.env` |
| 2 | Loaded TXT documents from `data/` |
| 3 | Split documents into overlapping chunks |
| 4 | Verified / created the Pinecone index |
| 5 | Embedded chunks and upserted them to Pinecone |
| 6 | *(Optional)* Connected to an existing vectorstore |
| 7 | Built the LCEL RAG chain |
| 8 | Ran example queries and viewed answers |
| 9 | Inspected retrieved source documents |

### Next Steps

- Add more TXT documents to `data/` and re-run from **Step 2**.
- Experiment with different `chunk_size`, `chunk_overlap`, and `top_k` values.
- Swap the LLM model (`gpt-4o`, `gpt-3.5-turbo`) or embedding model.
- Use different Pinecone namespaces to isolate document sets.